# `POST /validate-config` — assignment examples

Manual checks against the running server. Each cell sends one of the three input/output examples from `specs/assignment.md` to `POST /validate-config` and prints the prettified JSON response.

**Prereqs**

- Server running locally: `npm run dev:server` (or `npm run dev`).
- `OPENAI_API_KEY` set in `server/.env` (otherwise the endpoint returns `502`).
- Python `requests` available: `pip install requests`.

In [ ]:
import json
import requests

BASE_URL = "http://localhost:3000"
ENDPOINT = f"{BASE_URL}/validate-config"
MODEL = "gpt-5"  # e.g. "gpt-4o-mini" or "gpt-4o"; None uses the server default

def call_validate(config: dict, model: str | None = MODEL) -> None:
    """POST `config` to /validate-config and pretty-print the response."""
    params = {"model": model} if model else None
    print("Request:")
    print(json.dumps(config, indent=2))
    print()
    response = requests.post(ENDPOINT, json=config, params=params, timeout=120)
    print(f"HTTP {response.status_code}")
    try:
        body = response.json()
        print(json.dumps(body, indent=2, ensure_ascii=False))
    except ValueError:
        print(response.text)

## Example 1 — reward too high for an easy level

Expected pattern: schema is valid; the LLM should flag a `reward_vs_difficulty` mismatch (5000 reward on `easy`).

In [2]:
call_validate({
    "level": 12,
    "time_limit": 60,
    "reward": 5000,
    "difficulty": "easy"
})

Request:
{
  "level": 12,
  "time_limit": 60,
  "reward": 5000,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 12 (early game) is marked easy with a generous 60s timer, which fits the easy band. The reward is 5000, which sits at the top of the hard-tier reward range and far above easy’s allowance, creating a difficulty–reward mismatch.",
    "suggested_actions": [
      "Reduce reward to 100–500 for easy difficulty."
    ],
    "confidence": 0.98
  }
}


## Example 2 — time limit too tight for a hard level

Expected pattern: schema is valid; the LLM should flag `time_vs_difficulty` (10s on `hard`) and likely `frustration_risk`.

In [3]:
call_validate({
    "level": 5,
    "time_limit": 10,
    "reward": 500,
    "difficulty": "hard"
})

Request:
{
  "level": 5,
  "time_limit": 10,
  "reward": 500,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 5 is marked hard with a very tight 10s timer and a reward of 500. The timer sits at the hard band’s minimum, but the reward is far below the hard reward range (it aligns with easy/medium). Placing a hard-tier level this early in a 150-level game also suggests a progression spike.",
    "suggested_actions": [
      "Raise reward to 2000–5000 for hard difficulty.",
      "Re-tag this early level as easy/medium or move it to a later slot."
    ],
    "confidence": 0.88
  }
}


## Example 3 — reasonable starting level (expect empty findings)

Expected pattern: schema is valid; the LLM should return an empty `findings` array, so `suggested_actions` is `["No action needed"]` and `confidence` is the model's `verdict_confidence`.

In [4]:
call_validate({
    "level": 1,
    "time_limit": 120,
    "reward": 100,
    "difficulty": "easy"
})

Request:
{
  "level": 1,
  "time_limit": 120,
  "reward": 100,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 1 is marked easy with reward 100 and a very generous 120s timer. Both values sit within the easy ranges (reward at the floor; timer above the ≥30s minimum with no max). The early progression position aligns with an easy difficulty. No economy-wide distortion is implied by the minimal reward.",
    "suggested_actions": [
      "No action needed"
    ],
    "confidence": 0.97
  }
}


## Bonus — schema-validation failure (expect HTTP 400, no LLM call)

Sends a malformed body to confirm the Zod gate rejects it before the LLM is called.

In [5]:
call_validate({"level": "oops"})

Request:
{
  "level": "oops"
}

HTTP 400
{
  "schema_validation": {
    "valid": false,
    "errors": [
      {
        "path": "level",
        "message": "Expected number, received string"
      },
      {
        "path": "time_limit",
        "message": "Required"
      },
      {
        "path": "reward",
        "message": "Required"
      },
      {
        "path": "difficulty",
        "message": "Required"
      }
    ]
  }
}


## Example 4 — late-game level marked easy

Expected pattern: schema is valid; the LLM should flag `level_vs_difficulty`. Level 145 sits near the end of a 150-level game, so calling it `easy` breaks the typical early/mid/late progression even though reward and time stay inside the easy band.

In [ ]:
call_validate({
    "level": 145,
    "time_limit": 90,
    "reward": 300,
    "difficulty": "easy"
})

## Example 5 — generous timer on a hard level removes pressure

Expected pattern: schema is valid; the LLM should flag `time_vs_difficulty`. A 90s timer is well above the hard band's typical upper bound (30s), so the level no longer creates the intended challenge even though level number and reward fit `hard`.

In [ ]:
call_validate({
    "level": 140,
    "time_limit": 90,
    "reward": 4500,
    "difficulty": "hard"
})

## Example 6 — reasonable mid-game medium level (expect empty findings)

Expected pattern: schema is valid; the LLM should return an empty `findings` array. Level 75 is mid-progression in a 150-level game (aligning with `medium`), and reward (1200) and time (40s) both sit inside the medium band.

In [ ]:
call_validate({
    "level": 75,
    "time_limit": 40,
    "reward": 1200,
    "difficulty": "medium"
})

## Example 7 — multi-rule violation: tight timer + huge reward on early easy

Expected pattern: schema is valid; the LLM should fire several rules. The reward (4000) lives in the hard band on an `easy` level (`reward_vs_difficulty`); a 5s timer on `easy` is unusually tight (`time_vs_difficulty`, `frustration_risk`); reward-per-second is ~800 (`reward_per_second`); and the magnitude is exploitable (`economy_risk`).

In [ ]:
call_validate({
    "level": 8,
    "time_limit": 5,
    "reward": 4000,
    "difficulty": "easy"
})

## Example 8 — end-game hard at the boundary (expect empty findings)

Expected pattern: schema is valid; the LLM should return an empty `findings` array. Level 150 is the last level (`total_levels = 150`), the difficulty is `hard`, and reward + time both sit cleanly inside the hard band.

In [ ]:
call_validate({
    "level": 150,
    "time_limit": 25,
    "reward": 4500,
    "difficulty": "hard"
})